# Jigsaw Toxic Comment Classification — DistilBERT (Multi-Label)

Fine-tunes `distilbert-base-uncased` for **multi-label** classification: 6 independent
yes/no outputs per comment (`toxic`, `severe_toxic`, `obscene`, `threat`, `insult`,
`identity_hate`), not one-class-of-6.

### The key difference vs. a binary/multi-class setup
| | Binary / multi-class (wrong here) | Multi-label (correct) |
|---|---|---|
| Final layer | 1 or N outputs + **softmax** | **6 outputs + sigmoid** |
| Loss | Cross-entropy | **`BCEWithLogitsLoss`** (binary cross-entropy per label) |
| Labels can overlap? | No — picks exactly one class | **Yes** — any combination of the 6 can be 1 |

Softmax forces all outputs to sum to 1, which would wrongly mean "if this comment is
`toxic`, it can't also be `insult`." Sigmoid treats each of the 6 outputs as its own
independent probability, which is what we want.

We use HuggingFace's `Trainer` API so the training loop itself is just a few lines —
easier to read, easier to explain in a presentation, and well-tested.

### Before you start
Runtime ▸ Change runtime type ▸ **GPU (T4)** — this will be painfully slow on CPU.


## 1. Install & import

In [ ]:
!pip install -q transformers datasets accelerate evaluate


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU.")


## 2. Load data

Same files as the baselines notebook: `train.csv`, `test.csv`, `test_labels.csv`.


In [ ]:
from google.colab import files
uploaded = files.upload()  # select train.csv, test.csv, test_labels.csv


In [ ]:
# ---- Alternative: Kaggle API instead of manual upload ----
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle competitions download -c jigsaw-toxic-comment-classification-challenge
# !unzip -o jigsaw-toxic-comment-classification-challenge.zip -d data
# !unzip -o "data/train.csv.zip" -d data && unzip -o "data/test.csv.zip" -d data && unzip -o "data/test_labels.csv.zip" -d data


In [ ]:
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
TEST_LABELS_PATH = "test_labels.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
test_labels_df = pd.read_csv(TEST_LABELS_PATH)

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
NUM_LABELS = len(LABEL_COLS)

print("train shape:", train_df.shape)
train_df.head()


### Optional: subsample for fast iteration

DistilBERT fine-tuning on the full ~160k training rows takes a while even on a T4 GPU.
While you're getting the pipeline working (debugging, checking it runs end-to-end),
it's smart to train on a smaller slice first, then switch to the full dataset for your
final results. Set `USE_SUBSET = False` once you're ready for the real run.


In [ ]:
USE_SUBSET = True       # set to False for the final full-data run
SUBSET_SIZE = 20000     # rows to use while debugging

if USE_SUBSET:
    train_df = train_df.sample(n=SUBSET_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Using a subset of {len(train_df)} rows for fast iteration.")
else:
    print(f"Using full dataset: {len(train_df)} rows.")


## 3. Train / validation split

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["comment_text"].astype(str).tolist(),
    train_df[LABEL_COLS].values.astype(np.float32),
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("train size:", len(train_texts), " val size:", len(val_texts))


## 4. Tokenization

DistilBERT needs raw text turned into token IDs. We cap sequence length at 128 tokens —
long enough for almost all comments while keeping training fast. (Check the cell below;
if a lot of your comments are getting cut off, you can raise this to 192 or 256, at the
cost of slower training.)


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# Sanity check: what fraction of comments actually fit in MAX_LENGTH tokens?
sample_lengths = [len(tokenizer.encode(t)) for t in train_df["comment_text"].astype(str).sample(2000, random_state=RANDOM_STATE)]
pct_truncated = (np.array(sample_lengths) > MAX_LENGTH).mean() * 100
print(f"~{pct_truncated:.1f}% of a 2000-comment sample would be truncated at {MAX_LENGTH} tokens.")


In [ ]:
def tokenize(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)


## 5. Build HuggingFace `Dataset` objects

The `Trainer` API expects a `datasets.Dataset`, not raw lists/arrays. Labels must be
`float` (not `int`) because `BCEWithLogitsLoss` expects float targets.


In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels,  # already float32, shape (n_samples, 6)
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": val_labels,
})

train_dataset.set_format("torch")
val_dataset.set_format("torch")

train_dataset


## 6. Model — 6-output sigmoid head

`problem_type="multi_label_classification"` is the one line that makes HuggingFace
automatically use `BCEWithLogitsLoss` instead of cross-entropy. `num_labels=6` gives us
6 output neurons. There is **no softmax** anywhere here — `BCEWithLogitsLoss` applies
sigmoid internally per-label during the loss computation.


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",  # <-- this is the key line
)
model.to(device)
print("Model loaded with", NUM_LABELS, "output labels.")


## 7. Metric: mean column-wise ROC-AUC

Matches the competition's actual evaluation metric, and lets you compare directly
against your baselines and your teammate's BiLSTM.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)

    aucs = []
    for i in range(NUM_LABELS):
        # a label might have only one class present in a small eval batch/subset;
        # roc_auc_score errors in that case, so we skip it rather than crash
        if len(np.unique(labels[:, i])) > 1:
            aucs.append(roc_auc_score(labels[:, i], probs[:, i]))

    return {"mean_auc": float(np.mean(aucs)) if aucs else 0.0}


## 8. Training setup

Standard fine-tuning hyperparameters for DistilBERT: small learning rate, a couple of
epochs (BERT-family models overfit fast on this much data), and batch size sized for a
free-tier Colab GPU.


In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_jigsaw",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mean_auc",
    greater_is_better=True,
    report_to="none",  # disables wandb prompts
    fp16=torch.cuda.is_available(),  # mixed precision speeds up training on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


## 9. Train

In [ ]:
trainer.train()


## 10. Evaluate — per-label AUC breakdown

In [ ]:
val_predictions = trainer.predict(val_dataset)
val_probs = sigmoid(val_predictions.predictions)

print(f"{'Label':15s} {'AUC':>6s}")
per_label_aucs = []
for i, col in enumerate(LABEL_COLS):
    auc = roc_auc_score(val_labels[:, i], val_probs[:, i])
    per_label_aucs.append(auc)
    print(f"{col:15s} {auc:.4f}")

print(f"\n{'MEAN AUC':15s} {np.mean(per_label_aucs):.4f}")


In [ ]:
import matplotlib.pyplot as plt

pd.Series(per_label_aucs, index=LABEL_COLS).plot(
    kind="bar", figsize=(8, 4), title="DistilBERT — per-label AUC", ylim=(0.7, 1.0)
)
plt.ylabel("AUC")
plt.xticks(rotation=30)
plt.show()


## 11. Save the model (optional)

Useful if you want to reload it later without retraining, or hand it off to a teammate.


In [ ]:
model.save_pretrained("./distilbert_jigsaw_final")
tokenizer.save_pretrained("./distilbert_jigsaw_final")
print("Model and tokenizer saved to ./distilbert_jigsaw_final")

# To reload later:
# model = DistilBertForSequenceClassification.from_pretrained("./distilbert_jigsaw_final")
# tokenizer = DistilBertTokenizerFast.from_pretrained("./distilbert_jigsaw_final")


## 12. Predict on the real test set (Kaggle submission format, optional)

Note: `test_labels.csv` marks unscored rows with `-1` — those rows were excluded from
the competition's actual leaderboard. We still predict on all of `test.csv` for the
submission file, since that's what Kaggle's format expects.


In [ ]:
test_texts = test_df["comment_text"].astype(str).tolist()
test_encodings = tokenize(test_texts)

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
})
test_dataset.set_format("torch")

test_predictions = trainer.predict(test_dataset)
test_probs = sigmoid(test_predictions.predictions)

submission = pd.DataFrame(test_probs, columns=LABEL_COLS)
submission.insert(0, "id", test_df["id"])
submission.to_csv("submission_distilbert.csv", index=False)

submission.head()


## Summary — what makes this multi-label-correct

1. **`problem_type="multi_label_classification"`** swaps HuggingFace's internal loss to
   `BCEWithLogitsLoss` (sigmoid-based) instead of `CrossEntropyLoss` (softmax-based).
2. **Labels are float32 vectors of length 6** per example (e.g. `[1, 0, 1, 0, 1, 0]`),
   not a single integer class index.
3. **Evaluation uses per-label ROC-AUC, averaged** — directly comparable to your
   baselines notebook and your teammate's BiLSTM, since all three should report the
   same metric.

If your teammate's BiLSTM is using a `softmax` final activation or
`CategoricalCrossentropy`/`sparse_categorical_crossentropy` loss, flag it — it needs the
same sigmoid + binary-cross-entropy swap shown here to be valid for this task.
